In [2]:
import pandas as pd
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils import resample
import requests
import time
import numpy as np
import joblib

In [3]:
df = pd.read_csv('data/epi_r.csv', index_col=0)

In [4]:
df.head()

,rating,calories,protein,fat,sodium,#cakeweek,#wasteless,22-minute meals,3-ingredient recipes,30 days of groceries,...,yellow squash,yogurt,yonkers,yuca,zucchini,cookbooks,leftovers,snack,snack week,turkey
title,,,,,,,,,,,,,,,,,,,,,
"Lentil, Apple, and Turkey Wrap",2.500,426.0,30.0,7.0,559.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
Boudin Blanc Terrine with Red Onion Confit,4.375,403.0,18.0,23.0,1439.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Potato and Fennel Soup Hodge,3.750,165.0,6.0,7.0,165.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Mahi-Mahi in Tomato Olive Sauce,5.000,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Spinach Noodle Casserole,3.125,547.0,20.0,32.0,452.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [5]:
ingredients = [
    'milk', 'jam', 'almond', 'amaretto', 'anchovy', 'anise', 'apple', 'apple juice', 'apricot',
    'artichoke', 'arugula', 'asian pear', 'asparagus', 'avocado', 'bacon', 'banana',
    'barley', 'basil', 'bass', 'bean', 'beef', 'beef rib', 'beef shank', 'beef tenderloin',
    'beer', 'beet', 'bell pepper', 'berry', 'biscuit', 'bitters', 'blackberry', 'blue cheese',
    'blueberry', 'bok choy', 'bran', 'brandy', 'bread', 'breadcrumbs', 'brie', 'brisket',
    'broccoli', 'broccoli rabe', 'brown rice', 'brownie', 'brussel sprout', 'buffalo',
    'bulgur', 'burrito', 'butter', 'buttermilk', 'butternut squash', 'butterscotch/caramel',
    'cabbage', 'cake', 'candy', 'cantaloupe', 'capers', 'caraway', 'cardamom', 'carrot',
    'cashew', 'cauliflower', 'caviar', 'celery', 'chambord', 'champagne', 'chard',
    'cheddar', 'cheese', 'cherry', 'chestnut', 'chicken', 'chickpea', 'chile',
    'chile pepper', 'chili', 'chive', 'chocolate', 'cilantro', 'cinnamon', 'citrus',
    'clam', 'clove', 'coconut', 'cod', 'coffee', 'collard greens', 'cookie', 'cookies',
    'coriander', 'corn', 'cornmeal', 'cottage cheese', 'couscous', 'crab', 'cranberry',
    'cranberry sauce', 'cream cheese', 'créme de cacao', 'crêpe', 'cucumber', 'cumin',
    'cupcake', 'currant', 'curry', 'custard', 'dairy', 'date', 'dill', 'duck', 'egg',
    'egg nog', 'eggplant', 'endive', 'escarole', 'fennel', 'feta', 'fig', 'fish',
    'flat bread', 'fontina', 'fruit', 'fruit juice', 'garlic', 'ginger', 'goat cheese',
    'goose', 'gouda', 'grains', 'granola', 'grape', 'grapefruit', 'green bean',
    'green onion/scallion', 'ground beef', 'ground lamb', 'guava', 'halibut', 'ham',
    'hamburger', 'hazelnut', 'herb', 'hominy/cornmeal/masa', 'honey', 'honeydew',
    'horseradish', 'hummus', 'ice cream', 'jam or jelly', 'jalapeño', 'jícama', 'kale',
    'kiwi', 'kumquat', 'lamb', 'lamb chop', 'lamb shank', 'lasagna', 'leafy green',
    'leek', 'legume', 'lemon', 'lemon juice', 'lemongrass', 'lentil', 'lettuce',
    'lima bean', 'lime', 'lime juice', 'lingonberry', 'lobster', 'macadamia nut',
    'macaroni and cheese', 'mango', 'maple syrup', 'marinade', 'marsala', 'marshmallow',
    'mascarpone', 'mayonnaise', 'meat', 'meatball', 'meatloaf', 'melon', 'milk/cream',
    'mint', 'molasses', 'monterey jack', 'mozzarella', 'muffin', 'mushroom', 'mussel',
    'mustard', 'mustard greens', 'nectarine', 'noodle', 'nut', 'nutmeg', 'oat',
    'oatmeal', 'octopus', 'okra', 'olive', 'omelet', 'onion', 'orange', 'orange juice',
    'oregano', 'orzo', 'oyster', 'papaya', 'paprika', 'parmesan', 'parsley', 'parsnip',
    'pea', 'peach', 'peanut', 'peanut butter', 'pear', 'pecan', 'pepper', 'persimmon',
    'pine nut', 'pineapple', 'pistachio', 'plantain', 'plum', 'poblano', 'pomegranate',
    'pomegranate juice', 'poppy', 'pork', 'pork chop', 'pork rib', 'pork tenderloin',
    'potato', 'poultry', 'poultry sausage', 'prosciutto', 'prune', 'pumpkin', 'quail',
    'quiche', 'quince', 'quinoa', 'rabbit', 'radicchio', 'radish', 'raisin', 'raspberry',
    'red wine', 'rhubarb', 'rice', 'ricotta', 'rosemary', 'rosé', 'rum', 'rutabaga',
    'rye', 'saffron', 'sage', 'sake', 'salmon', 'salsa', 'sardine', 'sausage', 'scallop',
    'seafood', 'seed', 'semolina', 'sesame', 'sesame oil', 'shallot', 'shellfish',
    'sherry', 'shrimp', 'snapper', 'sorbet', 'sour cream', 'spinach', 'spice', 'squash',
    'squid', 'steak', 'strawberry', 'stuffing/dressing', 'sugar snap pea', 'sweet potato/yam',
    'swiss cheese', 'swordfish', 'tamarind', 'tangerine', 'tapioca', 'tarragon', 'tea',
    'thyme', 'tilapia', 'tofu', 'tomatillo', 'tomato', 'tortillas', 'trout', 'tuna',
    'turnip', 'turkey', 'vanilla', 'veal', 'vegetable', 'vinegar', 'walnut', 'wasabi',
    'watercress', 'watermelon', 'wheat/gluten-free', 'white wine', 'wild rice', 'wine',
    'yogurt', 'zucchini'
]


ingredients_df = df[[col for col in df.columns if col.lower() in ingredients]]


In [6]:
ingredients_df.to_csv('data/ingredients_df.csv')

In [7]:
X = ingredients_df
y = df['rating']

In [8]:
X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2, random_state=21)

In [10]:
param_grid = {
    'RandomForest': {
        'model': RandomForestRegressor(random_state=21),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, 20, None]
        }
    },
    'DecisionTreeRegressor': {
        'model': DecisionTreeRegressor(random_state=21),
        'params': {
            'max_depth': [3, 5, 10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4],
            'max_features': [None, 'sqrt', 'log2'],  
            'max_leaf_nodes': [None, 10, 20, 50],
            'min_weight_fraction_leaf': [0.0, 0.01, 0.05]
        }
    },
        'GradientBoostingRegressor': {
        'model': GradientBoostingRegressor(random_state=21),
        'params': {
            'n_estimators': [50, 100, 200],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5, 10]
        }
    }
}


best_model = {}

for model_name, param in param_grid.items():
    grid = GridSearchCV(
        estimator=param['model'],
        param_grid=param['params'],
        cv=3,
        scoring='neg_mean_squared_error',
        n_jobs=1
    )
    grid.fit(X_train, y_train)
    best_model[model_name] = grid.best_estimator_
    print(f"The best {model_name} params: {grid.best_params_}")


The best RandomForest params: {'max_depth': 20, 'n_estimators': 200}
The best DecisionTreeRegressor params: {'max_depth': None, 'max_features': 'sqrt', 'max_leaf_nodes': 50, 'min_samples_leaf': 4, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0}
The best GradientBoostingRegressor params: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}


In [11]:
for model_name, param in best_model.items():
    y_pred=param.predict(X_test)
    RMSE=np.sqrt(mean_squared_error(y_test, y_pred))
    print(f"{model_name} RMSE on test: {RMSE}")

RandomForest RMSE on test: 1.2845731499448294
DecisionTreeRegressor RMSE on test: 1.2883438225190473
GradientBoostingRegressor RMSE on test: 1.264806986037655


In [12]:
X = ingredients_df
y = df['rating'].round().astype(int)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [13]:
param_classifiers = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=5000),
        'params': {'C': [0.01, 0.1, 1, 10]}
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=21),
        'params': {'n_estimators': [50, 100], 'max_depth': [5, 10, 20]}
    },
    'SVC': {
        'model': SVC(),
        'params': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {'n_neighbors': [3, 5, 7, 9]}
    }
}

best_models = {}
for model, param in param_classifiers.items():
    clf = GridSearchCV(param['model'], param['params'], cv=5, scoring='accuracy', n_jobs=-1)
    clf.fit(X_train, y_train)
    best_models[model] = clf
    print(f"Best {model} params:", clf.best_params_)

Best LogisticRegression params: {'C': 0.1}
Best RandomForest params: {'max_depth': 20, 'n_estimators': 50}


/home/pagechiv/Desktop/pagechiv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best SVC params: {'C': 1, 'kernel': 'rbf'}
Best KNN params: {'n_neighbors': 9}


In [14]:
for name_model, param in best_models.items():
    y_pred = param.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name_model} Test Accuracy: {accuracy}")

LogisticRegression Test Accuracy: 0.6773871852405884
RandomForest Test Accuracy: 0.675392670157068
SVC Test Accuracy: 0.6798803290949887
KNN Test Accuracy: 0.62528047868362


In [15]:
def ratings(x):
    if x in [0,1]:
        return 'bad'
    elif x in [2,3]:
        return 'so-so'
    elif x in [4,5]:
        return 'great'

df['rating_category']=df['rating'].round().astype(int).apply(ratings)

In [16]:
X = ingredients_df
y=df['rating_category']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21)

In [17]:
param_cross = {
    'LogisticRegression': {
        'model': LogisticRegression(max_iter=5000),
        'params': {
            'C': [0.01, 0.1, 1, 10],
            'class_weight': ['balanced']  
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=21),
        'params': {
            'n_estimators': [50, 100],
            'max_depth': [5, 10, 20],
            'class_weight': ['balanced', 'balanced_subsample']
        }
    },
    'SVC': {
        'model': SVC(),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf'],
            'class_weight': ['balanced']
        }
    }
}

best_models = {}
for model, param in param_classifiers.items():
    clf = GridSearchCV(param['model'], param['params'], cv=5, scoring='accuracy', n_jobs=-1)
    clf.fit(X_train, y_train)
    best_models[model] = clf
    print(f"Best {model} params:", clf.best_params_)

Best LogisticRegression params: {'C': 1}
Best RandomForest params: {'max_depth': 20, 'n_estimators': 100}
Best SVC params: {'C': 1, 'kernel': 'rbf'}
Best KNN params: {'n_neighbors': 9}


In [18]:
for name_model, param in best_models.items():
    y_pred = param.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{name_model} Test Accuracy: {accuracy}")

LogisticRegression Test Accuracy: 0.800049862877088
RandomForest Test Accuracy: 0.8002991772625281
SVC Test Accuracy: 0.8010471204188482
KNN Test Accuracy: 0.7741211667913238


In [19]:
cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=21)


grid = GridSearchCV(
    estimator=RandomForestClassifier(random_state=21, class_weight='balanced_subsample'),
    param_grid={
        'n_estimators': [50, 100, 200],
        'max_depth': [10, 20, 30],
        'min_samples_leaf': [1, 2, 4]
    },
    scoring='f1_macro',
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best weighted F1:", grid.best_score_)

Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best params: {'max_depth': 30, 'min_samples_leaf': 1, 'n_estimators': 50}
Best weighted F1: 0.4416726465915402


In [20]:
report = classification_report(y_test, y_pred, output_dict=True)
df_report = pd.DataFrame(report).T

label_df = df_report[df_report.index.isin(['bad', 'great', 'so-so'])]

best_precision_label = label_df['precision'].idxmax()
best_precision_value = label_df['precision'].max()
support_value = label_df.loc[best_precision_label, 'support']


print(classification_report(y_test, y_pred, digits=3))
print(f"The best precision is in '{best_precision_label}' with {best_precision_value:.3f}")

              precision    recall  f1-score   support

         bad      0.300     0.245     0.270       372
       great      0.815     0.939     0.872      3200
       so-so      0.455     0.023     0.043       439

    accuracy                          0.774      4011
   macro avg      0.523     0.402     0.395      4011
weighted avg      0.728     0.774     0.726      4011

The best precision is in 'great' with 0.815


In [21]:
model_configs = {
    'RandomForestClassifier': {
        'model': RandomForestClassifier(random_state=21),
        'params': {
            'n_estimators': [50, 100],
            'max_depth': [20],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2],
            'max_features': ['sqrt'],
        }
    },
    'GradientBoostingClassifier': {
        'model': GradientBoostingClassifier(random_state=21),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.05, 0.1],
            'max_depth': [3, 5],
        }
    }
}

best_models = {}
for model, param in model_configs.items():
    bst = GridSearchCV(param['model'], param['params'], cv=5, scoring='accuracy', n_jobs=-1)
    bst.fit(X_train, y_train)
    best_models[model] = bst
    print(f"Best {model} params:", bst.best_params_)

Best RandomForestClassifier params: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best GradientBoostingClassifier params: {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 50}


In [22]:
best_model=RandomForestClassifier(class_weight=None, max_depth=20, max_features='sqrt', min_samples_leaf=1, min_samples_split=2, n_estimators=50, random_state=21)
best_model.fit(X_train, y_train)

y_pred=best_model.predict(X_test)

acc=accuracy_score(y_test, y_pred)

print(f"The accuarcy: {acc}%")

The accuarcy: 0.799800548491648%


In [9]:
y_pred=np.full_like(y_test, fill_value=y_train.mean())

RMSE=np.sqrt(mean_squared_error(y_test, y_pred))
RMSE

np.float64(1.307237065771332)

In [23]:
joblib.dump(best_model,'data/model.pkl')

['data/model.pkl']

In [24]:
API_KEY = "qchHPy2OFWPbSiZbehlbhh2DO639ubjQJBowezNp"

def search_ingredient(ingredient):
    url = f"https://api.nal.usda.gov/fdc/v1/foods/search"
    params = {
        "query": ingredient,
        "api_key": API_KEY,
        "pageSize": 1,
        "dataType": ["Survey (FNDDS)", "Foundation"]
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        foods = response.json().get("foods", [])
        if foods:
            return foods[0]["fdcId"]
    return None

In [25]:
def get_nutrition(fdc_id):
    url = f"https://api.nal.usda.gov/fdc/v1/food/{fdc_id}?api_key={API_KEY}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    return None


In [26]:
DAILY_VALUES = {
    "Total Fat": 78,
    "Saturated Fat": 20,
    "Cholesterol": 300,
    "Total Carbohydrate": 275,
    "Dietary Fiber": 28,
    "Total Sugars": 50,
    "Sodium": 2300,
    "Protein": 50,

    "Vitamin A": 900, 
    "Vitamin C": 90,
    "Calcium": 1300,
    "Iron": 18,
    "Vitamin D": 20, 
    "Vitamin E": 15,
    "Vitamin K": 120, 
    "Thiamin": 1.2,
    "Riboflavin": 1.3,
    "Niacin": 16,
    "Vitamin B6": 1.7,
    "Folate": 400,
    "Vitamin B12": 2.4,
    "Biotin": 30,
    "Pantothenic acid": 5,
    "Phosphorus": 1250,
    "Iodine": 150,
    "Magnesium": 420,
    "Zinc": 11,
    "Selenium": 55,
    "Copper": 0.9,
    "Manganese": 2.3,
    "Chromium": 35,
    "Molybdenum": 45,
    "Chloride": 2300,
    "Potassium": 4700,
    "Choline": 550
}



In [27]:
ingredients = list(ingredients_df.columns)
results = []

for ingredient in ingredients:
    fdc_id = search_ingredient(ingredient)
    if fdc_id:
        data = get_nutrition(fdc_id)
        nutrient_data = {"ingredient": ingredient}

        for nutrient in data.get("foodNutrients", []):
            name = nutrient["nutrient"]["name"]
            amount = nutrient.get("amount")
            unit = nutrient["nutrient"]["unitName"]

            if name in DAILY_VALUES and amount:
                daily_value = DAILY_VALUES[name]

                if unit.upper() in ["UG", "MCG"]:
                    amount = amount / 1000
                if unit.upper() == "MG":
                    amount = amount / 1000
                nutrient_data[name] = (amount / daily_value) * 100


                pct_dv = (amount / daily_value) * 100
                nutrient_data[name] = round(pct_dv, 2)
                
        results.append(nutrient_data)
        time.sleep(1)

df_nutrition = pd.DataFrame(results)


In [28]:
df_nutrition.to_csv("data/nutrition_facts_pct.csv", index=False)

In [29]:
df_nutrition.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288 entries, 0 to 287
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   ingredient        288 non-null    object 
 1   Protein           276 non-null    float64
 2   Total Sugars      237 non-null    float64
 3   Thiamin           266 non-null    float64
 4   Riboflavin        266 non-null    float64
 5   Niacin            266 non-null    float64
 6   Cholesterol       134 non-null    float64
 7   Biotin            5 non-null      float64
 8   Pantothenic acid  4 non-null      float64
dtypes: float64(8), object(1)
memory usage: 20.4+ KB


In [30]:
titles = df.index

def get_link_epicurious(title):
    query = title.replace(" ", "+")
    url = f"https://www.epicurious.com/search?q={query}"

    return url

recipe_links = []
for title in titles:
    link = get_link_epicurious(title)
    if link:
        recipe_links.append({'title': title, 'url': link})

links_df = pd.DataFrame(recipe_links)


In [31]:
links_df = links_df.reset_index(drop=True)
df = df.reset_index(drop=True)

links_df["rating"] = df["rating"]
links_df.to_csv("data/recipe_links.csv", index=False)

In [32]:
pd.read_csv('data/recipe_links.csv').sample(10)

,title,url,rating
15439,Grilled Lamb Kebabs with Coriander and Cumin,https://www.epicurious.com/search?q=Grilled+La...,4.375
19218,Basic Nut Milk,https://www.epicurious.com/search?q=Basic+Nut+...,4.375
17221,Beef Tagliata with Radicchio and Arugula,https://www.epicurious.com/search?q=Beef+Tagli...,3.750
19150,Mincemeat Ice Cream,https://www.epicurious.com/search?q=Mincemeat+...,4.375
8228,Braided Almond-Cream Wreath (Kranzkuchen),https://www.epicurious.com/search?q=Braided+Al...,0.000
6332,Ladies' Mile Applesauce Raisin Cake Diamond,https://www.epicurious.com/search?q=Ladies'+Mi...,3.750
4360,Chocolate-Amaretti Tortes,https://www.epicurious.com/search?q=Chocolate-...,4.375
3271,"Cashew Chard ""Burritos""",https://www.epicurious.com/search?q=Cashew+Cha...,4.375
8799,Brown Butter Pecan Shortbread,https://www.epicurious.com/search?q=Brown+Butt...,4.375
16179,Plum and Apricot Puff Pastry Tarts,https://www.epicurious.com/search?q=Plum+and+A...,4.375
